# PHASE 8 - Model Comparison

This notebook compares the metrics already produced by Linear Regression, Random Forest, XGBoost, and GRU. It does not retrain any model.

The model is selected using validation RMSE only. The test metrics are then reported as a final held-out comparison and are not used for selection.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_project_root():
    for candidate in [Path("."), Path("..")]:
        if (candidate / "results/metrics/linear_regression_metrics.csv").exists():
            return candidate
    raise FileNotFoundError("Model metric files were not found in results/metrics.")

project_root = find_project_root()
metrics_dir = project_root / "results/metrics"
figures_dir = project_root / "results/figures"
metric_files = {
    "Linear Regression": "linear_regression_metrics.csv",
    "Random Forest": "random_forest_metrics.csv",
    "XGBoost": "xgboost_metrics.csv",
    "GRU": "gru_metrics.csv",
}

metrics_frames = []
for model_name, filename in metric_files.items():
    path = metrics_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file: {path}")
    frame = pd.read_csv(path)
    frame["model"] = model_name
    metrics_frames.append(frame)

all_metrics = pd.concat(metrics_frames, ignore_index=True)
print(f"Loaded metrics for {all_metrics['model'].nunique()} models.")

## Comparison table

In [ ]:
validation_metrics = (
    all_metrics[all_metrics["split"] == "validation"]
    [["model", "MAE", "RMSE", "R2", "Training Time"]]
    .sort_values("RMSE")
    .reset_index(drop=True)
)
test_metrics = (
    all_metrics[all_metrics["split"] == "test"]
    [["model", "MAE", "RMSE", "R2", "Training Time"]]
    .set_index("model")
)

comparison_columns = {"model": "Model", "MAE": "MAE", "RMSE": "RMSE", "R2": "R²", "Training Time": "Training Time"}
validation_comparison = validation_metrics.rename(columns=comparison_columns)
test_comparison = (
    test_metrics.reindex(validation_metrics["model"])
    .reset_index()
    .rename(columns=comparison_columns)
)

display(validation_comparison.round(4))
display(test_comparison.round(4))
validation_comparison.to_csv(metrics_dir / "model_comparison_validation.csv", index=False)
test_comparison.to_csv(metrics_dir / "model_comparison_test.csv", index=False)

selected_model = validation_comparison.iloc[0]["Model"]
selected_test = test_comparison[test_comparison["Model"] == selected_model].iloc[0]
selection = pd.DataFrame([{
    "selected_by": "validation RMSE",
    "Model": selected_model,
    "validation_RMSE": validation_comparison.iloc[0]["RMSE"],
    "test_MAE": selected_test["MAE"],
    "test_RMSE": selected_test["RMSE"],
    "test_R²": selected_test["R²"],
}])
selection.to_csv(metrics_dir / "selected_model.csv", index=False)
print(f"Selected model by validation RMSE: {selected_model}")
print(f"Held-out test RMSE for selected model: {selected_test['RMSE']:.4f}")

In [ ]:
validation_metrics = (
    all_metrics[all_metrics["split"] == "validation"]
    [["model", "MAE", "RMSE", "R2", "Training Time"]]
    .sort_values("RMSE")
    .reset_index(drop=True)
)
validation_comparison = validation_metrics.rename(columns=comparison_columns)
display(validation_comparison.round(4))
validation_comparison.to_csv(metrics_dir / "model_comparison_validation.csv", index=False)

In [ ]:
plot_data = test_comparison.set_index("Model")[["MAE", "RMSE"]]
ax = plot_data.plot(kind="bar", figsize=(11, 5))
ax.set_title("Test-set error comparison (ordered by validation RMSE)")
ax.set_ylabel("Error")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(figures_dir / "model_comparison_test_errors.png", dpi=150)
plt.show()

## Phase 8 conclusion

The comparison is based on saved test metrics, not assumptions. PHASE 9 can analyze the selected model's errors by hour, weekday, month, season, and working-day status.